# Expected Shortfall (CVaR)

VaR (05/01) answers "how bad is the loss at the edge of the tail?" and stays silent about the
tail itself. Expected shortfall answers "**if** we are in the tail, how much do we lose on
average?". It is an expected value of a function of the loss, it has a closed form for the
normal and Student-$t$, it is subadditive, and it can be minimized with a linear program. Since
2016 it is the Basel market-risk capital measure.

## 0. Math link

- [Expected value of a function](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/03_expected_value_of_a_function.ipynb):
  ES is $E[L \mid L \ge \operatorname{VaR}]$, and the Rockafellar–Uryasev formula writes it as
  $\min_c\, \{c + E[(L - c)^+]/(1-\alpha)\}$, an expected value of a function of $L$.
- [Quantiles](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/12_quantiles.ipynb):
  ES is the average of the VaRs beyond the confidence level.
- [u-substitution](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/04_integrals/03_u_substitution.ipynb):
  the normal closed form comes from $\int x\,\varphi(x)\,dx = -\varphi(x)$.
- VaR, the fat-tailed simulation and the two-bond example from [05/01 Value at Risk](./01_value_at_risk.ipynb).

## 1. Introduction

Two trading desks report the same 99% VaR of CHF 1 million. One loses at most about CHF 2.4 million
on its worst days; the other has a scenario in which it loses CHF 9 million. VaR cannot tell them
apart. Expected shortfall can.

After this notebook you can:

- define ES and compute it for discrete, historical, normal and Student-$t$ losses;
- derive the normal closed form $\operatorname{ES}_\alpha = \sigma\,\varphi(z_\alpha)/(1-\alpha)$;
- explain why ES is subadditive and VaR is not;
- use the Rockafellar–Uryasev formula to find a minimum-ES portfolio by linear programming;
- explain why ES is harder to estimate and to backtest than VaR.

## 2. Theory

### Definition

For a loss $L$ (CHF, positive = loss) with continuous distribution and confidence level $\alpha$:

$$
\operatorname{ES}_\alpha = E\left[L \mid L \ge \operatorname{VaR}_\alpha\right]
= \frac{1}{1-\alpha}\int_\alpha^1 \operatorname{VaR}_u\,du
$$

The second form holds for **any** distribution and is the general definition. It follows from
writing $L = F^{-1}(U)$ with $U$ uniform on $(0,1)$: the event $L \ge \operatorname{VaR}_\alpha$ is $U \ge \alpha$,
which has probability $1-\alpha$, and $F^{-1}(u) = \operatorname{VaR}_u$. ES is the **average VaR over all
confidence levels beyond $\alpha$**, so $\operatorname{ES}_\alpha \ge \operatorname{VaR}_\alpha$ always.

Other names: conditional VaR (CVaR), average VaR, tail VaR (the last is sometimes defined
slightly differently for discrete losses).

### Discrete losses

For $n$ equally likely scenarios with $(1-\alpha)n = k$ an integer, ES is the average of the $k$
worst losses. If $(1-\alpha)n$ is not an integer, the boundary scenario gets fractional weight
so that exactly a share $1-\alpha$ of probability is averaged:

$$
\operatorname{ES}_\alpha = \frac{1}{1-\alpha}\Big(E\big[L\,\mathbf 1\{L > \operatorname{VaR}_\alpha\}\big]
+ \operatorname{VaR}_\alpha\,\big(P(L \le \operatorname{VaR}_\alpha) - \alpha\big)\Big)
$$

### Normal losses

If $L = \sigma Z$ with $Z \sim N(0, 1)$ and $z_\alpha = \Phi^{-1}(\alpha)$:

$$
E[Z \mid Z \ge z_\alpha] = \frac{1}{1-\alpha}\int_{z_\alpha}^\infty x\,\varphi(x)\,dx
= \frac{1}{1-\alpha}\Big[-\varphi(x)\Big]_{z_\alpha}^\infty = \frac{\varphi(z_\alpha)}{1-\alpha}
$$

using $\varphi'(x) = -x\,\varphi(x)$ (substitute $u = x^2/2$). So, for a position of value $V$ with
return volatility $\sigma$ and mean $\mu$:

$$
\operatorname{ES}_\alpha = V\left(\sigma\,\frac{\varphi(z_\alpha)}{1-\alpha} - \mu\right)
$$

| $\alpha$ | 95% | 97.5% | 99% | 99.5% |
|---|---|---|---|---|
| VaR multiplier $z_\alpha$ | 1.645 | 1.960 | 2.326 | 2.576 |
| ES multiplier $\varphi(z_\alpha)/(1-\alpha)$ | 2.063 | 2.338 | 2.665 | 2.892 |

The 97.5% ES multiplier (2.338) is almost the 99% VaR multiplier (2.326). Basel chose 97.5% ES to
replace 99% VaR for this reason: **for normal returns**, capital stays about the same; for
fat-tailed returns it goes up.

### Student-$t$ losses

For a standard $t_\nu$ variable with density $g_\nu$ and quantile $t_\alpha$:

$$
E[T \mid T \ge t_\alpha] = \frac{g_\nu(t_\alpha)}{1-\alpha}\cdot\frac{\nu + t_\alpha^2}{\nu - 1}
$$

multiplied by $\sigma\sqrt{(\nu-2)/\nu}$ when the $t$ is scaled to volatility $\sigma$ (as in 05/01). For
$\nu = 4$ the 97.5% ES multiplier is 2.82, against 2.65 for the 99% VaR and 2.34 for the normal ES.

### Rockafellar–Uryasev: ES as a minimum

Define for a threshold $c$

$$
F_\alpha(c) = c + \frac{1}{1-\alpha}\,E\big[(L - c)^+\big]
$$

an expected value of the function $\ell \mapsto (\ell - c)^+$. $F_\alpha$ is convex in $c$, and for a
continuous loss distribution

$$
F_\alpha'(c) = 1 - \frac{P(L > c)}{1-\alpha} = 0 \iff P(L > c) = 1 - \alpha \iff c = \operatorname{VaR}_\alpha
$$

At the minimum, $F_\alpha(\operatorname{VaR}_\alpha) = \operatorname{VaR}_\alpha + E[L - \operatorname{VaR}_\alpha \mid L > \operatorname{VaR}_\alpha] = \operatorname{ES}_\alpha$. So

$$
\operatorname{ES}_\alpha = \min_c F_\alpha(c), \qquad \operatorname{VaR}_\alpha = \arg\min_c F_\alpha(c)
$$

With scenarios $L_1,\dots,L_n$ the expectation is an average, $(L_j - c)^+$ becomes an auxiliary
variable $u_j \ge L_j - c,\ u_j \ge 0$, and **minimizing ES over portfolio weights is a linear
program**. Minimizing VaR over weights, by contrast, is a non-convex problem.

### Coherence

Artzner, Delbaen, Eber and Heath (1999) list four properties a risk measure $\rho$ should have:

1. **Monotonicity**: if $L_1 \le L_2$ in every scenario, then $\rho(L_1) \le \rho(L_2)$.
2. **Translation invariance**: $\rho(L + c) = \rho(L) + c$ for a sure amount $c$.
3. **Positive homogeneity**: $\rho(\lambda L) = \lambda\,\rho(L)$ for $\lambda \ge 0$.
4. **Subadditivity**: $\rho(L_1 + L_2) \le \rho(L_1) + \rho(L_2)$: merging does not create risk.

VaR satisfies 1–3 but not 4 (05/01, Example 4). ES satisfies all four. Proof of subadditivity for
continuous losses: ES is the largest average loss over any event $A$ of probability $1-\alpha$,

$$
\operatorname{ES}_\alpha(L) = \max_{P(A) = 1-\alpha} E[L \mid A]
$$

because the tail event $\{L \ge \operatorname{VaR}_\alpha\}$ collects the largest losses. Take $A^*$, the tail event of
$L_1 + L_2$. Then

$$
\operatorname{ES}_\alpha(L_1 + L_2) = E[L_1 \mid A^*] + E[L_2 \mid A^*] \le \operatorname{ES}_\alpha(L_1) + \operatorname{ES}_\alpha(L_2)
$$

since each $E[L_i \mid A^*]$ is an average over one particular event of probability $1-\alpha$, at
most the maximum over all of them.

### The price: estimation and backtesting

- ES depends on the **whole tail**, including the few most extreme observations. Its sampling error
  is larger than that of VaR at the same $\alpha$.
- VaR is backtested by counting exceptions (05/01). ES has no equally simple test: it is not
  *elicitable* (no scoring function is minimized by the true ES, Gneiting 2011), so Basel backtests
  ES models with VaR exceptions at 97.5% and 99%.

## 3. Worked Examples

### Example 1: a single position, normal returns

CHF 10,000,000, daily volatility 1%, mean ignored (05/01, Example 1).

| | 95% | 97.5% | 99% |
|---|---|---|---|
| VaR | CHF 164,500 | CHF 196,000 | CHF 232,600 |
| ES | $2.063 \cdot 100{,}000 =$ CHF 206,300 | CHF 233,800 | CHF 266,500 |

On the 1% worst days the average loss is CHF 266,500, about 15% above the 99% VaR.

### Example 2: same VaR, different ES

Two desks are each evaluated on 1,000 equally likely scenarios. The ten worst losses (CHF million):

| Desk | ten worst losses | 99% VaR (10th worst) | 99% ES (average of ten) |
|---|---|---|---|
| A | 2.4, 2.0, 1.8, 1.6, 1.5, 1.4, 1.3, 1.2, 1.1, 1.0 | 1.0 | 15.3 / 10 = **1.53** |
| B | 9.0, 6.0, 4.0, 2.5, 1.5, 1.2, 1.1, 1.1, 1.1, 1.0 | 1.0 | 28.5 / 10 = **2.85** |

Taking the 10th worst loss as the VaR is one convention (05/01 Example 3 discusses others). Desk B's
tail is nearly twice as costly. A VaR limit of CHF 1 million treats the desks as equal.

### Example 3: the two bonds, now with ES

Two independent bonds of CHF 1,000,000, each defaulting with probability 4% (05/01, Example 4).

**One bond, 95% ES.** The worst 5% of outcomes consist of the 4% default (loss CHF 1,000,000) and 1%
of the no-default outcomes (loss 0):

$$
\operatorname{ES}_{95\%} = \frac{0.04 \cdot 1{,}000{,}000 + 0.01 \cdot 0}{0.05} = \text{CHF } 800{,}000
$$

**Both bonds.** $P(\text{2 defaults}) = 0.04^2 = 0.16\%$, $P(\text{exactly 1}) = 2 \cdot 0.04 \cdot 0.96 = 7.68\%$.
The worst 5% are the 0.16% double default (CHF 2,000,000) and 4.84% of the single defaults:

$$
\operatorname{ES}_{95\%} = \frac{0.0016 \cdot 2{,}000{,}000 + 0.0484 \cdot 1{,}000{,}000}{0.05} = \text{CHF } 1{,}032{,}000
$$

$1{,}032{,}000 \le 800{,}000 + 800{,}000$: ES rewards the diversification that VaR punished
(VaR: CHF 0 + 0 alone, CHF 1,000,000 together).

### Example 4: Rockafellar–Uryasev by hand

Losses of CHF 0, 1, 2, 3, 10 (thousand), each with probability 20%. At $\alpha = 80\%$ the tail is
the single worst outcome, so $\operatorname{ES}_{80\%} = 10$ and $\operatorname{VaR}_{80\%} = 3$. Check with $F_{0.8}(c) = c + 5\,E[(L - c)^+]$:

- $c = 3$: $E[(L-3)^+] = 0.2 \cdot 7 = 1.4$, so $F = 3 + 7 = 10$.
- $c = 2$: $E[(L-2)^+] = 0.2 \cdot (1 + 8) = 1.8$, so $F = 2 + 9 = 11$.
- $c = 5$: $E[(L-5)^+] = 0.2 \cdot 5 = 1$, so $F = 5 + 5 = 10$.

The minimum is 10 = ES. For this discrete distribution $F$ is flat between 3 and 10, so every
$c$ in $[3, 10]$ is a minimizer; the smallest one is the VaR.

## 4. Finance Application

### Who uses ES

- **Banks (Basel FRTB, in force in the EU and Switzerland from the mid-2020s)**: market-risk
  capital under internal models is a 97.5% ES, computed on a 10-day base horizon and scaled up for
  less liquid risk factors, calibrated to a stress period. VaR remains for backtesting.
- **Insurers**: the Swiss Solvency Test (FINMA) uses the 99% one-year ES of risk-bearing capital;
  Solvency II uses 99.5% VaR.
- **Asset managers**: mean-CVaR optimization (Rockafellar–Uryasev) for portfolios with
  options, credit or other asymmetric payoffs, where variance misses the tail.

### Reading the ES/VaR ratio

For a normal distribution $\operatorname{ES}_{99\%}/\operatorname{VaR}_{99\%} = 2.665 / 2.326 = 1.15$, and the ratio
falls toward 1 as $\alpha \to 1$. For a Student-$t$ with $\nu$ degrees of freedom it tends to
$\nu/(\nu - 1)$, 1.33 for $\nu = 4$. A reported ratio well above 1.15 at 99% says the tail is fatter
than normal, or that the portfolio contains jump risk (short options, credit).

### The usual misreadings

- "ES is the worst case." It is the **average** loss in the worst $1-\alpha$ share of outcomes;
  some of those losses are larger, and ES does not say by how much.
- "ES and VaR rank portfolios the same way." Example 2 and the put-writing asset below show they need
  not.
- "A historical ES from one year of data is reliable." At 97.5% it averages the six or seven worst
  days of the year; one more crash day moves it a lot.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate, optimize, sparse

rng = np.random.default_rng(42)
trading_days = 252
value = 10_000_000   # CHF

### Closed forms and three checks

Normal and Student-$t$ ES, checked against (a) the integral of VaR over $[\alpha, 1]$, (b) simulation,
and (c) the Rockafellar–Uryasev minimum.

In [ ]:
def es_normal(value, sigma, alpha, mu=0.0):
    # ES in currency units, positive = loss
    return value * (sigma * stats.norm.pdf(stats.norm.ppf(alpha)) / (1 - alpha) - mu)


def es_t(value, sigma, alpha, df):
    # Student-t scaled to volatility sigma
    t_a = stats.t.ppf(alpha, df)
    es_std = stats.t.pdf(t_a, df) / (1 - alpha) * (df + t_a**2) / (df - 1)
    return value * sigma * np.sqrt((df - 2) / df) * es_std


def var_t(value, sigma, alpha, df):
    return value * sigma * np.sqrt((df - 2) / df) * stats.t.ppf(alpha, df)


sigma, alpha, df_t = 0.01, 0.99, 4

es_n_integral = value * sigma * integrate.quad(stats.norm.ppf, alpha, 1)[0] / (1 - alpha)
es_t_integral = integrate.quad(lambda u: var_t(value, sigma, u, df_t), alpha, 1, limit=200)[0] / (1 - alpha)

n_sims = 2_000_000
loss_n = value * sigma * rng.standard_normal(n_sims)
loss_t = value * sigma * np.sqrt((df_t - 2) / df_t) * rng.standard_t(df_t, n_sims)


def es_historical(losses, alpha):
    # average of the worst (1 - alpha) share; the boundary scenario gets fractional weight
    losses = np.sort(np.asarray(losses))[::-1]
    k = (1 - alpha) * len(losses)
    k_int = int(np.floor(k))
    total = losses[:k_int].sum() + (k - k_int) * (losses[k_int] if k_int < len(losses) else 0.0)
    return total / k


check = pd.DataFrame({
    "closed form": [es_normal(value, sigma, alpha), es_t(value, sigma, alpha, df_t)],
    "integral of VaR": [es_n_integral, es_t_integral],
    "simulation (2m draws)": [es_historical(loss_n, alpha), es_historical(loss_t, alpha)],
}, index=["normal", "t(4)"])
check.style.format("CHF {:,.0f}")

In [ ]:
def ru_objective(c, losses, alpha):
    return c + np.mean(np.maximum(losses - c, 0.0)) / (1 - alpha)


for name, losses in [("normal", loss_n), ("t(4)", loss_t)]:
    res = optimize.minimize_scalar(ru_objective, args=(losses, alpha),
                                   bounds=(0, 6 * value * sigma), method="bounded")
    print(f"{name:6s}: argmin c = CHF {res.x:,.0f}  (99% VaR CHF {np.quantile(losses, alpha):,.0f}), "
          f"min F = CHF {res.fun:,.0f}  (99% ES CHF {es_historical(losses, alpha):,.0f})")

The minimizer of $F_\alpha$ is the VaR and the minimum value is the ES, as derived.

### Worked examples in code

In [ ]:
# Example 1
for a in [0.95, 0.975, 0.99]:
    print(f"Example 1, {a:.1%}: VaR CHF {value * sigma * stats.norm.ppf(a):,.0f}, ES CHF {es_normal(value, sigma, a):,.0f}")

# Example 2: 990 smaller losses plus the ten worst, in CHF million
desk_tails = {"A": [2.4, 2.0, 1.8, 1.6, 1.5, 1.4, 1.3, 1.2, 1.1, 1.0],
              "B": [9.0, 6.0, 4.0, 2.5, 1.5, 1.2, 1.1, 1.1, 1.1, 1.0]}
body = np.linspace(-2.0, 0.99, 990)
for desk, tail in desk_tails.items():
    scen = np.concatenate([body, tail])
    print(f"Example 2, desk {desk}: 10th worst CHF {np.sort(scen)[-10]:.2f}m, 99% ES CHF {es_historical(scen, 0.99):.2f}m")

In [ ]:
def es_discrete(values, probs, alpha):
    # exact ES for a discrete loss distribution, using the formula from the Theory section
    values, probs = np.asarray(values, float), np.asarray(probs, float)
    order = np.argsort(values)
    values, probs = values[order], probs[order]
    cdf = np.cumsum(probs)
    var = values[np.searchsorted(cdf, alpha - 1e-12)]
    above = values > var
    es = (np.sum(values[above] * probs[above]) + var * (cdf[values <= var][-1] - alpha)) / (1 - alpha)
    return var, es


# Example 3: one bond and two bonds
p, face = 0.04, 1_000_000
print("Example 3, one bond:  VaR CHF {:,.0f}, ES CHF {:,.0f}".format(*es_discrete([0, face], [1 - p, p], 0.95)))
two = stats.binom.pmf([0, 1, 2], 2, p)
print("Example 3, two bonds: VaR CHF {:,.0f}, ES CHF {:,.0f}".format(*es_discrete([0, face, 2 * face], two, 0.95)))

# Example 4
print("Example 4: VaR {:.0f}, ES {:.0f}".format(*es_discrete([0, 1, 2, 3, 10], [0.2] * 5, 0.80)))

### ES/VaR ratios: normal vs fat tails

In [ ]:
rows = []
for nu in [3, 4, 6, 10, 30, np.inf]:
    name = "normal" if np.isinf(nu) else f"t({nu})"
    var99 = stats.norm.ppf(0.99) if np.isinf(nu) else var_t(1, 1, 0.99, nu)
    es975 = es_normal(1, 1, 0.975) if np.isinf(nu) else es_t(1, 1, 0.975, nu)
    es99 = es_normal(1, 1, 0.99) if np.isinf(nu) else es_t(1, 1, 0.99, nu)
    rows.append({"distribution": name, "99% VaR / σ": var99, "97.5% ES / σ": es975,
                 "99% ES / σ": es99, "ES99 / VaR99": es99 / var99, "ES97.5 / VaR99": es975 / var99})
ratio_table = pd.DataFrame(rows).set_index("distribution")
ratio_table.round(3)

For the normal, moving from 99% VaR to 97.5% ES changes capital by half a percent. For a $t(4)$ it
adds 6.6%, for a $t(3)$ 11%. The fatter the tail, the more the switch costs.

### Estimation noise: VaR vs ES from one year of data

Draw 5,000 one-year histories (252 days) of $t(4)$ returns with 1% daily volatility. Estimate the
99% VaR, the 97.5% ES and the 99% ES from each and compare the spread with the true value.

In [ ]:
n_repeats, n_days = 5_000, trading_days
samples = value * sigma * np.sqrt((df_t - 2) / df_t) * rng.standard_t(df_t, (n_repeats, n_days))

estimates = pd.DataFrame({
    "99% VaR": [np.quantile(s, 0.99) for s in samples],
    "97.5% ES": [es_historical(s, 0.975) for s in samples],
    "99% ES": [es_historical(s, 0.99) for s in samples],
})
true_values = pd.Series({
    "99% VaR": var_t(value, sigma, 0.99, df_t),
    "97.5% ES": es_t(value, sigma, 0.975, df_t),
    "99% ES": es_t(value, sigma, 0.99, df_t),
})
noise = pd.DataFrame({
    "true (CHF)": true_values,
    "mean estimate (CHF)": estimates.mean(),
    "std / true": estimates.std() / true_values,
    "5% quantile / true": estimates.quantile(0.05) / true_values,
    "95% quantile / true": estimates.quantile(0.95) / true_values,
})
noise.style.format({"true (CHF)": "{:,.0f}", "mean estimate (CHF)": "{:,.0f}", "std / true": "{:.1%}",
                    "5% quantile / true": "{:.2f}", "95% quantile / true": "{:.2f}"})

All three estimates are biased low with one year of fat-tailed data, and the 99% ES the most
(about 5%): most years contain no observation from the far tail, so the average of the worst 2.5
days misses the rare very large losses that pull the true ES up. Its relative standard error is
28%, against 17.5% for the 99% VaR. The 97.5% ES averages about 6 days and its error (20%) is close
to that of the 99% VaR, which is one reason Basel chose 97.5% rather than 99%.

### Concentration: VaR and ES of a bond portfolio

CHF 1,000,000 spread equally over $n$ independent bonds, each defaulting with probability 4%.
The number of defaults is $\operatorname{Binomial}(n, 0.04)$, so the loss distribution is exact.

In [ ]:
n_bonds = np.arange(1, 41)
concentration = []
for n in n_bonds:
    k = np.arange(n + 1)
    var95, es95 = es_discrete(k * face / n, stats.binom.pmf(k, n, p), 0.95)
    concentration.append({"bonds": n, "95% VaR": var95, "95% ES": es95})
concentration = pd.DataFrame(concentration).set_index("bonds")
concentration.loc[[1, 2, 3, 5, 10, 20, 40]].style.format("CHF {:,.0f}")

VaR jumps from CHF 0 to CHF 500,000 when the million is split over two bonds, falls in steps, and
then **rises** again when a further bond is added (from about CHF 111,000 with 9 bonds to CHF 200,000
with 10), whenever one more default enters the worst 5%. ES falls steadily from CHF 800,000: more names, smaller tail. A VaR
limit would tell the manager to hold a single bond.

### Minimum-variance vs minimum-ES portfolios

Add a fourth asset to the 05/01 universe: a **put-writing** strategy. It collects a premium every
day with low volatility, and once in roughly two years it loses 6% in a day. Simulate 20,000 daily
scenarios: the three original assets from the multivariate $t(4)$, the put writer independent.

In [ ]:
def multivariate_t(n, Sigma, df, rng):
    # same generator as 05/01
    L = np.linalg.cholesky(Sigma * (df - 2) / df)
    z = rng.standard_normal((n, Sigma.shape[0])) @ L.T
    chi2 = rng.chisquare(df, size=(n, 1))
    return z / np.sqrt(chi2 / df)


assets = ["Swiss equities", "Global equities", "Swiss bonds", "Put-writing"]
vols = np.array([0.16, 0.17, 0.05])                  # annual
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma_daily = np.diag(vols) @ corr @ np.diag(vols) / trading_days
mu_daily = np.array([0.065, 0.07, 0.02]) / trading_days

n_scen = 20_000
p_crash, crash = 0.002, -0.06                        # daily crash probability, crash return
put_r = (0.04 / trading_days - p_crash * crash       # premium: 4% a year net of expected crash loss
         + 0.003 * rng.standard_normal(n_scen)
         + crash * (rng.random(n_scen) < p_crash))
R = np.column_stack([mu_daily + multivariate_t(n_scen, Sigma_daily, df_t, rng), put_r])

stand_alone = pd.DataFrame({
    "annual vol": R.std(axis=0, ddof=1) * np.sqrt(trading_days),
    "99% VaR (% of value)": [np.quantile(-R[:, i], 0.99) for i in range(4)],
    "99% ES (% of value)": [es_historical(-R[:, i], 0.99) for i in range(4)],
    "normal ES, same vol": [es_normal(1, R[:, i].std(ddof=1), 0.99) for i in range(4)],
}, index=assets)
print(f"crash days in the sample: {np.sum(put_r < -0.04)}")
stand_alone.style.format("{:.2%}")

The put writer has the lowest 99% VaR of all four assets, below even its own normal approximation:
the crash days fall beyond the 1% quantile, so VaR does not see them. Its ES is well above the
normal ES for its volatility. Now minimize variance and, separately, 99% ES, long-only and fully
invested. The ES problem is the Rockafellar–Uryasev linear program: variables $(w, c, u_1, \dots, u_n)$,

$$
\min\; c + \frac{1}{(1-\alpha)n}\sum_j u_j \quad\text{s.t.}\quad u_j \ge -R_j^\top w - c,\; u_j \ge 0,\; \textstyle\sum_i w_i = 1,\; w \ge 0
$$

In [ ]:
def min_variance_weights(R):
    S = np.cov(R, rowvar=False)
    k = R.shape[1]
    res = optimize.minimize(lambda x: x @ S @ x, np.full(k, 1 / k), method="SLSQP",
                            bounds=[(0, 1)] * k,
                            constraints=[{"type": "eq", "fun": lambda x: x.sum() - 1}],
                            options={"ftol": 1e-14, "maxiter": 500})
    return res.x


def min_es_weights(R, alpha):
    n, k = R.shape
    # variables: w (k), c (1), u (n)
    cost = np.concatenate([np.zeros(k), [1.0], np.full(n, 1 / ((1 - alpha) * n))])
    # -R w - c - u <= 0, sparse because of the n x n identity block
    A_ub = sparse.hstack([sparse.csr_matrix(-R), sparse.csr_matrix(-np.ones((n, 1))), -sparse.eye(n)])
    A_eq = np.concatenate([np.ones(k), [0.0], np.zeros(n)])[None, :]
    bounds = [(0, 1)] * k + [(None, None)] + [(0, None)] * n
    res = optimize.linprog(cost, A_ub=A_ub, b_ub=np.zeros(n), A_eq=A_eq, b_eq=[1.0],
                           bounds=bounds, method="highs")
    return res.x[:k], res.fun


w_mv = min_variance_weights(R)
w_es, es_min = min_es_weights(R, 0.99)

def describe(w):
    r = R @ w
    return {"annual vol": r.std(ddof=1) * np.sqrt(trading_days),
            "annual mean": r.mean() * trading_days,
            "99% VaR": np.quantile(-r, 0.99),
            "99% ES": es_historical(-r, 0.99)}

comparison = pd.DataFrame({
    "min variance": list(w_mv) + list(describe(w_mv).values()),
    "min 99% ES": list(w_es) + list(describe(w_es).values()),
}, index=assets + ["annual vol", "annual mean", "99% VaR (daily)", "99% ES (daily)"])
print(f"LP optimum (99% ES, daily): {es_min:.3%}")
comparison.style.format("{:.2%}")

The minimum-variance portfolio puts about a third of its weight in the put writer: low volatility,
no correlation with anything. The minimum-ES portfolio cuts it to about 12% and holds more Swiss
bonds. It accepts a higher volatility (4.2% vs 3.9% a year) and even a higher 99% VaR
(0.72% vs 0.60% a day) for a smaller average loss on the worst 1% of days (0.98% vs 1.12%).
By VaR alone the minimum-variance portfolio looks safer, and the put writer, with the lowest
stand-alone VaR of the four, looks like the safest asset.
Variance treats a steady 0.3% wobble and a rare 6% crash as interchangeable sources of the same
variance; ES does not.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
x = np.linspace(-4, 7, 1_000)
t_scale = np.sqrt((df_t - 2) / df_t)
for ax, (name, pdf, var_m, es_m) in zip(axes, [
        ("normal", stats.norm.pdf(x), stats.norm.ppf(0.99), es_normal(1, 1, 0.99)),
        ("t(4), same σ", stats.t.pdf(x / t_scale, df_t) / t_scale, var_t(1, 1, 0.99, df_t), es_t(1, 1, 0.99, df_t))]):
    ax.plot(x, pdf, color="black")
    tail = x >= var_m
    ax.fill_between(x[tail], pdf[tail], color="tab:red", alpha=0.4, label="worst 1%")
    ax.axvline(var_m, color="tab:blue", linestyle="--", label=f"99% VaR = {var_m:.2f} σ")
    ax.axvline(es_m, color="tab:red", label=f"99% ES = {es_m:.2f} σ")
    ax.set_yscale("log")
    ax.set_ylim(1e-5, 1)
    ax.set_xlim(-4, 7)
    ax.set_title(f"Daily loss, {name}")
    ax.set_xlabel("loss (in units of daily σ)")
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
alphas = np.linspace(0.90, 0.999, 200)
plt.figure(figsize=(8, 5))
plt.plot(alphas, es_normal(1, 1, alphas) / stats.norm.ppf(alphas), color="black", label="normal")
for nu, color in [(3, "tab:red"), (4, "tab:orange"), (10, "tab:blue")]:
    plt.plot(alphas, es_t(1, 1, alphas, nu) / var_t(1, 1, alphas, nu), color=color, label=f"t({nu})")
    plt.axhline(nu / (nu - 1), color=color, linestyle=":", linewidth=1)
plt.title("ES / VaR at the same confidence level (dotted: limit ν/(ν−1))")
plt.xlabel("confidence level α")
plt.ylabel("ES / VaR")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Rockafellar-Uryasev objective on the simulated t(4) losses
c_grid = np.linspace(0, 6 * value * sigma, 120)
F = [ru_objective(c, loss_t[:200_000], 0.99) for c in c_grid]
var99_t, es99_t = np.quantile(loss_t, 0.99), es_historical(loss_t, 0.99)

plt.figure(figsize=(8, 5))
plt.plot(c_grid, F, color="black", label="F(c) = c + E[(L − c)⁺] / (1 − α)")
plt.axvline(var99_t, color="tab:blue", linestyle="--", label="99% VaR (minimizer)")
plt.axhline(es99_t, color="tab:red", linestyle=":", label="99% ES (minimum)")
plt.title("ES as the minimum of an expected value, t(4) losses, CHF 10m at 1% daily vol")
plt.xlabel("threshold c (CHF)")
plt.ylabel("F(c) (CHF)")
fmt = plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f}k")
plt.gca().xaxis.set_major_formatter(fmt)
plt.gca().yaxis.set_major_formatter(fmt)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.step(concentration.index, concentration["95% VaR"], where="mid", color="tab:blue", label="95% VaR")
plt.plot(concentration.index, concentration["95% ES"], "o-", color="tab:red", markersize=3, label="95% ES")
plt.title("CHF 1m over n independent bonds, default probability 4% each")
plt.xlabel("number of bonds n")
plt.ylabel("risk (CHF)")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f}k"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for col, color in [("99% VaR", "tab:blue"), ("97.5% ES", "tab:green"), ("99% ES", "tab:red")]:
    axes[0].hist(estimates[col] / true_values[col], bins=80, range=(0.4, 2.0), histtype="step",
                 linewidth=1.5, color=color, label=col)
axes[0].axvline(1, color="black", linestyle="--")
axes[0].set_title("One-year historical estimates / true value, t(4)")
axes[0].set_xlabel("estimate / true")
axes[0].set_ylabel("count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x_pos = np.arange(len(assets))
axes[1].bar(x_pos - 0.2, comparison.loc[assets, "min variance"], width=0.4, label="min variance")
axes[1].bar(x_pos + 0.2, comparison.loc[assets, "min 99% ES"], width=0.4, label="min 99% ES")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(assets, rotation=15)
axes[1].set_title("Portfolio weights")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. The CHF 5,000,000 bond portfolio of 05/01 (daily volatility 0.3%): compute its 97.5% and 99%
   1-day ES under normality and compare with the 99% VaR.
2. With the ten daily returns of 05/01 Example 3 (%: $+0.4, -1.2, +0.8, -0.3, -2.5, +1.1, -0.6, +0.2, -1.8, +0.5$)
   and a CHF 1,000,000 portfolio, compute the 80% and 85% historical ES by hand, then check with
   `es_historical`.

### Intermediate

3. A loss $L$ is uniform on $[0, 1]$ (CHF million). Derive $\operatorname{VaR}_\alpha$ and $\operatorname{ES}_\alpha$, then confirm
   numerically that $F_\alpha(c)$ is minimized at $c = \operatorname{VaR}_\alpha$ for $\alpha = 0.9$.
4. Interpret: a desk reports a 99% VaR of CHF 2 million and a 97.5% ES of CHF 3 million. What does
   the ES number mean in words? Under normality, what 97.5% ES would you expect for that VaR, and
   what does the difference suggest about the desk's positions?

### Advanced

5. Show that if returns are multivariate normal with mean zero, the minimum-ES and the
   minimum-variance portfolios coincide for every $\alpha$. Then explain, in two or three sentences,
   which assumption fails for the put-writing asset and why the two optimizers disagree.

## 8. Solutions / Checks

In [ ]:
# 1.
v1, s1 = 5_000_000, 0.003
print(f"1. 97.5% ES CHF {es_normal(v1, s1, 0.975):,.0f}, 99% ES CHF {es_normal(v1, s1, 0.99):,.0f}, "
      f"99% VaR CHF {v1 * s1 * stats.norm.ppf(0.99):,.0f}")

# 2.
r10 = np.array([0.4, -1.2, 0.8, -0.3, -2.5, 1.1, -0.6, 0.2, -1.8, 0.5]) / 100
for a in [0.80, 0.85]:
    print(f"2. {a:.0%} ES: CHF {es_historical(-1_000_000 * r10, a):,.0f}")

# 3.
a = 0.9
u_sims = rng.random(1_000_000)
res = optimize.minimize_scalar(ru_objective, args=(u_sims, a), bounds=(0, 1), method="bounded")
print(f"3. argmin c = {res.x:.4f} (VaR = {a}), min F = {res.fun:.4f} (ES = {(1 + a) / 2})")

# 4.
print(f"4. normal 97.5% ES for a 99% VaR of CHF 2m: CHF {2e6 * es_normal(1, 1, 0.975) / stats.norm.ppf(0.99):,.0f}")

**1.** 97.5% ES CHF 35,100, 99% ES CHF 40,000, 99% VaR CHF 34,900. The 97.5% ES and 99% VaR
almost coincide, as the normal multiplier table predicts.

**2.** Losses sorted: 2.5%, 1.8%, 1.2%, ... At 80% the tail is the 2 worst of 10:
$(2.5 + 1.8)/2 = 2.15\%$, CHF 21,500. At 85% the tail is 1.5 observations: the worst at full weight
and the second at half weight, $(2.5 + 0.5 \cdot 1.8)/1.5 = 2.27\%$, CHF 22,700.

**3.** $P(L \le \ell) = \ell$, so $\operatorname{VaR}_\alpha = \alpha$, and $\operatorname{ES}_\alpha = E[L \mid L \ge \alpha] = (1 + \alpha)/2$.
For $\alpha = 0.9$: VaR 0.9, ES 0.95 (CHF million). The numerical minimizer of $F_{0.9}$ lands at
0.9 with minimum 0.95.

**4.** On the worst 2.5% of days, the desk loses CHF 3 million on average. Under normality a 99%
VaR of CHF 2 million implies $\sigma = 2/2.326$ and a 97.5% ES of $2.338\,\sigma \approx$ CHF 2.01 million. The
reported CHF 3 million is about 50% higher: the loss distribution has a much fatter tail than the
normal, the signature of short options, credit or other jump exposure. Ask what the worst
scenarios are.

**5.** With mean zero, $R_p = R^\top w \sim N(0, w^\top\Sigma w)$, so $\operatorname{ES}_\alpha(w) = V\,k_\alpha\sqrt{w^\top\Sigma w}$ with
$k_\alpha = \varphi(z_\alpha)/(1-\alpha) > 0$, a strictly increasing function of the variance. Minimizing one minimizes
the other, under any constraints. With non-zero means $\operatorname{ES} = V(k_\alpha\sigma_p - \mu_p)$ and the two can
differ slightly through $\mu_p$, but for daily horizons that effect is small. The put writer's returns
are **not normal**: they are a mixture with a 0.2% chance of a 6% loss. Variance counts that crash
only through its contribution to $\sigma^2$, which is small because the crash is rare; ES averages
the worst 1% of days, where the crash dominates.

## 9. Conclusion

- $\operatorname{ES}_\alpha$ is the average loss in the worst $1-\alpha$ share of outcomes, equivalently the average of $\operatorname{VaR}_u$ for $u \ge \alpha$.
- Normal: $\operatorname{ES}_\alpha = \sigma\,\varphi(z_\alpha)/(1-\alpha)$; 97.5% ES $\approx$ 99% VaR, which is why Basel picked 97.5%.
- Fat tails raise ES relative to VaR; the ratio tends to $\nu/(\nu - 1)$ for a Student-$t$.
- ES is coherent: subadditive, so diversification never increases it.
- Rockafellar–Uryasev: $\operatorname{ES}_\alpha = \min_c\,\{c + E[(L-c)^+]/(1-\alpha)\}$, which makes minimum-ES portfolios a linear program.
- The price: ES is noisier to estimate than VaR and harder to backtest.

Next: [03 Drawdowns](./03_drawdowns.ipynb).